# TuringBench: Llama-4-Scout-Instruct LoRA Fine-tuning
Fine-tune Meta's latest **Llama-4-Scout-Instruct** (mixture-of-experts model) on TuringBench for human-vs-AI text classification.
Uses memory-efficient 4-bit bitsandbytes NF4 quantization + PEFT/LoRA on Kaggle T4 GPU.

In [ ]:
import os
import shutil

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print('Could not load HF_TOKEN from Kaggle Secrets:', e)
    print('Set HF_TOKEN manually or add it via Add-ons -> Secrets.')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/kaggle/working/ai-detection-at-scale'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone $repo_url $repo_dir
print('Repo ready at', repo_dir)
!git -C $repo_dir log -1 --oneline

In [ ]:
# Install dependencies including bitsandbytes and peft
!pip install -q datasets transformers accelerate scikit-learn huggingface_hub packaging bitsandbytes peft

In [ ]:
import os

script = os.path.join(repo_dir, 'scripts', '33_finetune_turingbench.py')
out_dir = '/kaggle/working/models/turingbench_llama4_scout'
os.makedirs(out_dir, exist_ok=True)

hub_model_id = 'vedangvatsa/vedang-turingbench-llama-4-scout'

# Resume from the latest Hub checkpoint if one exists and HF_TOKEN is set.
resume_from = None
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import list_repo_refs
        refs = list(list_repo_refs(repo_id=hub_model_id).branches)
        if refs:
            resume_from = hub_model_id
            print('Resuming from Hub checkpoint:', resume_from)
    except Exception as e:
        print('No existing Hub checkpoint found, starting fresh:', e)

# Run the fine-tuning script with Llama 4 Scout in 4-bit quantization using PEFT/LoRA!
cmd = (
    f"python {script} "
    f"--model_name meta-llama/Llama-4-Scout-17B-16E-Instruct "
    f"--output_dir {out_dir} "
    f"--max_length 256 "
    f"--epochs 1 "
    f"--batch_size 4 "
    f"--gradient_accumulation_steps 8 "
    f"--learning_rate 2e-4 "
    f"--load_in_4bit "
    f"--hub_model_id {hub_model_id} "
)
if resume_from:
    cmd += f"--resume_from_checkpoint {resume_from} "
cmd += "--seed 42"

print('Running:', cmd)
!{cmd}

In [ ]:
import shutil
results_src = os.path.join(repo_dir, 'results', 'turingbench_finetuned.csv')
results_dst = '/kaggle/working/results/turingbench_finetuned.csv'
os.makedirs('/kaggle/working/results', exist_ok=True)
if os.path.exists(results_src):
    shutil.copy(results_src, results_dst)
    print('Results copied to', results_dst)
print('Outputs in /kaggle/working/models/turingbench_llama4_scout')